# Text in boxes — MOSS-Image layout generation

You give a scene description plus a list of boxes: each box holds either **exact text to
render** or an **object to draw**. [MOSS-Image](https://github.com/OpenMOSS/MOSS-Image)
(5B image model, Apache-2.0 code, no account needed) generates the image with each item
placed in its box.

**Runtime:** *Runtime → Change runtime type* →
- **A100** — recommended, fastest.
- **L4** — works; slower because model parts are swapped between CPU and GPU memory.
- **T4** — not supported (too little memory, no fast bfloat16).

Then *Runtime → Run all*. The first run downloads ~30 GB of weights (several minutes).

**Text accuracy:** the model is strong at text but not flawless. Short phrases in roomy boxes
work best; use `\n` in `text` to force line breaks. The notebook makes one image per seed —
pick the cleanest, or add seeds.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


In [ ]:
# Official MOSS-Image code (pipeline helpers + memory offload); installs diffusers/transformers versions it needs.
!git clone -q --depth 1 https://github.com/OpenMOSS/MOSS-Image.git /content/MOSS-Image 2>/dev/null || git -C /content/MOSS-Image pull -q
!pip install -q -e /content/MOSS-Image


## Settings


In [ ]:
MODEL = "OpenMOSS-Team/MOSS-Image-SFT"   # or "OpenMOSS-Team/MOSS-Image-Base"
WIDTH, HEIGHT = 1024, 1024               # multiples of 16; 1–2 megapixels works best
STEPS = 50                               # official recommendation: 50 or more
CFG = 3.0                                # 3–6; raise a little if text/boxes are ignored
SEEDS = [42, 1234]                       # one image per seed
NEGATIVE = "worst quality, bad anatomy, watermark, jpeg artifact"


## Layout

Boxes are `(x1, y1, x2, y2)` in **ordinary pixels**: (0, 0) is the top-left corner,
(WIDTH, HEIGHT) the bottom-right. Each item is either
`{"text": ..., "box": ..., "style": ...}` or `{"object": ..., "box": ...}`.
Edit this cell to make your own image.


In [ ]:
SCENE = "A minimalist travel poster with a cream background and a flat blue mountain illustration."
KIND = "poster"   # e.g. poster, photo, illustration, infographic, ui

ITEMS = [
    {"text": "EXPLORE", "box": (128, 96, 896, 224),
     "style": "Large dark-blue bold uppercase title, centered."},
    {"object": "A flat blue mountain range illustration with a pale sun.",
     "box": (160, 288, 864, 816)},
    {"text": "THE ALPS\nSUMMER 2027", "box": (256, 848, 768, 960),
     "style": "Small dark-blue sans-serif caption, two centered lines."},
]


In [ ]:
import json

def to_moss_layout(scene, items, width, height, kind="poster"):
    """Convert top-left pixel boxes (x1, y1, x2, y2) to MOSS-Image's layout JSON.

    MOSS wants [y_min, x_min, y_max, x_max] in pixels relative to the canvas centre
    ("bbox_format": "yxyx", "bbox_coord_space": "abs_centralized").
    """
    assert width % 16 == 0 and height % 16 == 0, "WIDTH and HEIGHT must be multiples of 16"
    out = []
    for i, item in enumerate(items):
        x1, y1, x2, y2 = item["box"]
        assert 0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height, f"item {i}: box {item['box']} is outside the canvas"
        bbox = [round(y1 - height / 2), round(x1 - width / 2), round(y2 - height / 2), round(x2 - width / 2)]
        if "text" in item:
            out.append({"label": "text", "caption": item.get("style", "Clear, legible text."),
                        "text": item["text"], "bbox": bbox})
        else:
            out.append({"label": "object", "caption": item["object"], "bbox": bbox})
    return {"summary": {"meta": [kind], "canvas": "single", "caption": scene},
            "items": out, "bbox_coord_space": "abs_centralized", "bbox_format": "yxyx"}

layout = to_moss_layout(SCENE, ITEMS, WIDTH, HEIGHT, KIND)
PROMPT = json.dumps(layout, ensure_ascii=False)
print(json.dumps(layout, indent=1, ensure_ascii=False))


In [ ]:
# Preview the boxes before spending GPU time (red = text, blue = object).
from PIL import Image, ImageDraw

def draw_boxes(image, items):
    image = image.convert("RGB").copy()
    d = ImageDraw.Draw(image)
    for item in items:
        color = (220, 40, 40) if "text" in item else (40, 90, 220)
        d.rectangle(item["box"], outline=color, width=4)
        label = item.get("text") or item["object"]
        d.text((item["box"][0] + 6, item["box"][1] + 4), label[:40].replace("\n", " / "), fill=color,
               font_size=max(16, WIDTH // 40))
    return image

display(draw_boxes(Image.new("RGB", (WIDTH, HEIGHT), "white"), ITEMS))


## Load the model


In [ ]:
import sys, torch
sys.path.insert(0, "/content/MOSS-Image")
from inference_runtime import configure_pipeline, cuda_budget_gib, load_pipeline, set_scheduler

gib = cuda_budget_gib()
assert gib > 0, "No GPU: Runtime > Change runtime type > A100 or L4"
assert torch.cuda.is_bf16_supported(), "This GPU has no bfloat16 support (T4?). Switch to an L4 or A100."
# GPU-resident when it fits (~30 GB of weights); otherwise move whole components on demand.
OFFLOAD = "none" if gib >= 38 else "model" if gib >= 20 else "hybrid"
print(f"GPU memory {gib:.0f} GiB -> offload mode: {OFFLOAD}")

pipe = load_pipeline(MODEL)
set_scheduler(pipe, shift=5.0)
configure_pipeline(pipe, offload=OFFLOAD)


## Generate


In [ ]:
import os
os.makedirs("/content/outputs", exist_ok=True)

images = []
for seed in SEEDS:
    image = pipe(
        prompt=PROMPT,
        negative_prompt=NEGATIVE,
        height=HEIGHT,
        width=WIDTH,
        num_inference_steps=STEPS,
        guidance_scale=CFG,
        uncond_skip_layers=[22],
        generator=torch.Generator("cpu").manual_seed(seed),
    ).images[0]
    path = f"/content/outputs/layout_seed{seed}.png"
    image.save(path)
    images.append(image)
    print(f"seed {seed} -> {path}")
    display(image)


In [ ]:
# Same images with the requested boxes drawn on top, to check placement.
for seed, image in zip(SEEDS, images):
    print(f"seed {seed}")
    display(draw_boxes(image, ITEMS))


Images are saved in `/content/outputs` (📁 Files panel → right-click → Download).
To try another design, edit the **Layout** cell and run it and the cells below it — the model
stays loaded, so you don't need to re-run *Load the model*.
